# Lab 1, part A: the tools

Lab 1 (A+B) Costs: $0.099

Scenario 4, developer productivity. Fernhill keeps what its engineers need to know in three
places: a ticket queue, a service catalogue, and a set of architecture decisions. This part
builds an MCP server over all three, gives its failures a shape an agent can act on, and
wires it into Claude Code.

Part A makes no API calls. Everything here runs offline, and the proof that it works is a
terminal session at the end rather than a Python call.

Part B, after Chapter 7, hands the same server to an Agent SDK consumer.

## 1. A workspace to serve

Nothing here is Claude specific. It is the material the tools will expose: three tickets, a
service catalogue, two architecture decision records, and three small Python files.

The Python files carry two deliberate traps, and part B collects on both of them. One
function is re-exported under two other names, so searching for the original name finds a
third of the callers. And one line appears twice, identically, which is what makes an edit
anchor ambiguous.

In [ ]:
import json
from pathlib import Path

LAB = Path.cwd()
CODE = LAB.parent
if not (CODE / "pyproject.toml").exists():
    raise SystemExit(
        f"Run this notebook from its own folder inside labs/code. The working "
        f"directory is {LAB}, and {CODE / 'pyproject.toml'} is not there. "
        f"In JupyterLab the working directory follows the notebook, so open it "
        f"from the file browser rather than starting the kernel elsewhere."
    )

WORKSPACE = LAB / "workspace"

TICKETS = [
    {"id": "TKT-0042", "status": "open", "service": "refunds",
     "title": "Customers charged twice when a refund is retried",
     "body": "Two customers report duplicate charges after a refund was retried. "
             "The retry count in the refunds client looks wrong."},
    {"id": "TKT-0051", "status": "open", "service": "checkout",
     "title": "Order note validator rejects valid notes",
     "body": "Notes containing an apostrophe are rejected. Probably the validator base class."},
    {"id": "TKT-0038", "status": "closed", "service": "billing",
     "title": "Gateway timeouts during the Friday batch",
     "body": "Resolved by raising the gateway timeout. See ADR-0005."},
]

SERVICES = [
    {"name": "refunds", "owner": "Payments", "on_call": "Ada Okafor", "entrypoint": "shop/refunds.py"},
    {"name": "billing", "owner": "Payments", "on_call": "Ada Okafor", "entrypoint": "shop/billing.py"},
    {"name": "checkout", "owner": "Storefront", "on_call": "Nils Brandt", "entrypoint": "shop/legacy.py"},
]

FILES = {
    "data/tickets.json": json.dumps(TICKETS, indent=2),
    "data/services.json": json.dumps(SERVICES, indent=2),

    "data/adr/ADR-0003-money-as-floats.md": '''
# ADR-0003: Money is never a float

Status: accepted

Amounts are integers of pence, everywhere. Floats were the cause of three reconciliation
incidents in the first year. Any function that takes an amount takes `amount_pence: int`.
''',

    "data/adr/ADR-0005-payment-gateway-retries.md": '''
# ADR-0005: The gateway is retried at most twice

Status: accepted

The payment gateway is retried on timeout, at most twice, with a delay between attempts.
Retrying more than that has caused duplicate charges, because the gateway may have
succeeded without us hearing about it.
''',

    "shop/billing.py": '''
# Card handling. The one place a card is actually charged.


def charge_card(order_id: str, amount_pence: int) -> dict:
    # Every payment in the shop ends up here.
    return {"order_id": order_id, "amount_pence": amount_pence, "status": "charged"}
''',

    "shop/legacy.py": '''
# Kept for the old admin panel. See ADR-0004.

from billing import charge_card as take_payment

__all__ = ["take_payment"]
''',

    "shop/refunds.py": '''
# Refund client.

from billing import charge_card as process_payment


def send_refund(order_id: str, amount_pence: int) -> dict:
    max_retries = 3
    return process_payment(order_id, -amount_pence)


def send_reversal(order_id: str, amount_pence: int) -> dict:
    max_retries = 3
    return process_payment(order_id, -amount_pence)
''',
}

for relative, body in FILES.items():
    path = WORKSPACE / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(body.lstrip("\n"), encoding="utf-8")

print(f"workspace at {WORKSPACE}")
for relative in FILES:
    print(f"  {relative}")

## 2. What a tool returns when it fails

A tool has two jobs. Return an answer, and, when it cannot, return a failure the agent can
act on. The second one is the one that gets skipped.

`isError` on the result says only that something broke. On its own it leaves the agent
guessing between four different recoveries, two of which are expensive: retry a policy wall
forever, or abandon a timeout that would have worked on the next attempt.

So the payload carries the category, whether a retry can help, and what to do instead.

| Category | Retryable | What the agent should do |
|---|---|---|
| transient | yes | retry, with a delay |
| validation | yes, but only after the input changes | fix the arguments, then retry |
| business | no | explain it to the person, in their terms |
| permission | no | escalate |

And the case that is not a failure at all: a query that ran correctly and matched nothing is
a success with an empty list. Mark it as an error and the agent retries a question that was
already answered.

The next cell writes that helper.

In [ ]:
import json

ERRORS_PY = '''
# Structured failures. Raising is what sets isError on the result; returning a dictionary
# that happens to describe a failure is reported as a success.

import json

RETRYABLE = {"transient": True, "validation": True, "business": False, "permission": False}


class ToolError(ValueError):
    # FastMCP turns a raised exception into a result with isError set.
    pass


def tool_error(category: str, message: str, next_action: str, **extra) -> ToolError:
    if category not in RETRYABLE:
        raise KeyError(f"unknown category {category!r}")
    payload = {
        "errorCategory": category,
        "isRetryable": RETRYABLE[category],
        "message": message,
        "nextAction": next_action,
        **extra,
    }
    return ToolError(json.dumps(payload))
'''

path = WORKSPACE / "servers" / "errors.py"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(ERRORS_PY.lstrip("\n"), encoding="utf-8")
print(f"wrote {path.relative_to(WORKSPACE)}")

## 3. The five shapes, side by side

Four failures and one success, laid out together so the differences are visible at a glance.
The fifth row is the one worth pausing on: it is not an error, and the agent must not treat
it as one.

In [ ]:
import sys
sys.path.insert(0, str(WORKSPACE / "servers"))

from errors import tool_error

cases = [
    tool_error("transient", "The ticket store did not respond in time.",
               "Retry in a few seconds."),
    tool_error("validation", "No service called 'refund'. Expected one of: refunds, billing, checkout.",
               "Call again with one of the listed names."),
    tool_error("business", "That decision record is superseded and is not served any more.",
               "Tell the engineer it was superseded by ADR-0005, and read that instead."),
    tool_error("permission", "The ticket queue is readable only by the Payments group.",
               "Ask a Payments engineer, or escalate."),
]

print(f"{'category':12} {'retry?':7} next action")
print("-" * 76)
for case in cases:
    payload = json.loads(str(case))
    print(f"{payload['errorCategory']:12} {str(payload['isRetryable']):7} {payload['nextAction']}")

empty = {"query": "kubernetes", "matched": 0, "results": []}
print()
print("and the one that is not a failure:")
print(f"{'(no error)':12} {'n/a':7} {json.dumps(empty)}")

## 4. The server

An MCP server is a process that exposes tools. `FastMCP` turns decorated functions into
them, and `Field(description=...)` on each argument is what gives Claude a per argument
description in the generated schema.

Two things to notice in what gets written below, because part B is going to collect on
both.

The descriptions are thin, and the first tool is worse than thin: it is called `lookup` and
it says `Look things up.` Nothing there tells anyone, human or model, that this is the
support ticket queue. That is not a style problem, it is the selection mechanism, and part B
measures exactly what it costs.

And `architecture_decisions` is doing two jobs: listing what exists, and fetching one. A
tool that does two things is harder to describe than either of them separately.

In [ ]:
SERVER_PY = '''
# Developer productivity tools over the Fernhill workspace.

import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP
from pydantic import Field

from errors import tool_error

DATA = Path(__file__).resolve().parent.parent / "data"
mcp = FastMCP("devtools")


@mcp.tool(description="Look things up.")
def lookup(query: str = Field(description="Search text")) -> dict:
    tickets = json.loads((DATA / "tickets.json").read_text())
    hits = [t for t in tickets
            if query.lower() in (t["title"] + " " + t["body"]).lower()]
    return {"query": query, "matched": len(hits), "results": hits}


@mcp.tool(description="Get service info.")
def service_catalogue(name: str = Field(description="Service name")) -> dict:
    services = json.loads((DATA / "services.json").read_text())
    for service in services:
        if service["name"] == name:
            return service
    known = ", ".join(s["name"] for s in services)
    raise tool_error(
        "validation",
        f"No service called {name!r}. Expected one of: {known}.",
        "Call again with one of the listed names.",
    )


@mcp.tool(description="Look up decisions.")
def architecture_decisions(
    adr_id: str = Field(default="", description="An ADR id, or empty to list them all"),
) -> dict:
    records = sorted(DATA.glob("adr/ADR-*.md"))
    if not adr_id:
        return {"available": [p.stem for p in records]}
    for path in records:
        if path.stem.startswith(adr_id):
            return {"id": path.stem, "text": path.read_text()}
    raise tool_error(
        "validation",
        f"No decision record {adr_id!r}.",
        "Call again with no id to list the ids that exist.",
    )


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

path = WORKSPACE / "servers" / "devtools_server.py"
path.write_text(SERVER_PY.lstrip("\n"), encoding="utf-8")
print(f"wrote {path.relative_to(WORKSPACE)}")

## 5. Call the tools directly

No model involved. A decorated FastMCP tool is still an ordinary function, so the fastest
way to know the server works is to call it.

Three outcomes below: a hit, a query that matched nothing, and a failure that raises.

One wrinkle worth knowing, because it will bite you the first time you improvise here. Pass
every argument explicitly. An argument you leave out does not fall back to its default, it
arrives as the `Field` descriptor itself, and the tool fails on a type nobody expected.

In [ ]:
import importlib
import devtools_server
importlib.reload(devtools_server)
from devtools_server import lookup, service_catalogue, architecture_decisions
from errors import ToolError

hit = lookup(query="duplicate charges")
print(f"hit          matched {hit['matched']}: {hit['results'][0]['id']} {hit['results'][0]['title']}")

nothing = lookup(query="kubernetes")
print(f"no matches   matched {nothing['matched']}, results {nothing['results']}, and no error raised")

print(f"catalogue    {service_catalogue(name='refunds')}")
print(f"decisions    {architecture_decisions(adr_id='')['available']}")

try:
    service_catalogue(name="refund")
except ToolError as exc:
    print(f"failure      {exc}")

## 6. What the model actually sees

This is the whole basis on which Claude chooses. Not the code, not the data: the name, the
description, and the argument descriptions.

Read the three descriptions below as if you were the one choosing. Given a repository you
can also search with `Grep`, is there anything here that would make you pick these?

In [ ]:
tools = await devtools_server.mcp.list_tools()

for tool in tools:
    print(f"{tool.name}")
    print(f"    description: {tool.description}")
    for argument, schema in tool.inputSchema["properties"].items():
        print(f"    {argument}: {schema.get('description', '(none)')}")
    print()

## 7. Two scopes, both loaded at once

Claude Code reads server definitions from two places, and loads both at the same time.

**Project scope** is `.mcp.json` at the project root, in version control, for the servers the
whole team needs. Because it is committed, a credential never goes in it literally: write
`${DEVTOOLS_TOKEN}` and each engineer supplies their own from the environment. `${VAR:-default}`
falls back when the variable is unset.

**User scope** is your own file, for personal and experimental servers, added with
`claude mcp add --scope user`. It follows you between projects and nobody else sees it.

Once connected, the tools arrive named `mcp__<server>__<tool>`, so the server this cell
registers gives Claude `mcp__devtools__search_tickets`.

In [ ]:
config = {
    "mcpServers": {
        "devtools": {
            "command": "uv",
            "args": ["run", "--project", "../..", "python", "servers/devtools_server.py"],
            "env": {"DEVTOOLS_TOKEN": "${DEVTOOLS_TOKEN:-local-dev}"},
        }
    }
}

path = WORKSPACE / ".mcp.json"
path.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
print(path)
print(path.read_text())

## 8. Now prove it, in Claude Code

The notebook has not spoken to a model once. This is where that happens, and Claude Code is
the client.

Open a terminal in the workspace directory this notebook just created:

```bash
cd workspace
claude
```

**One.** Run `/mcp`. The `devtools` server should be listed and connected. That is project
scope, from the `.mcp.json` written above.

**Two.** Ask it something the server can answer:

> which open tickets mention duplicate charges?

Watch which tool it reaches for, and write it down. There is a `Grep` available to it and a
`search_tickets` described as "Search tickets." Whichever wins, that observation is what
part B starts from.

**Three.** Add the same server again under user scope, to see both at once:

```bash
claude mcp add --scope user devtools-personal -- uv run --project ../.. python servers/devtools_server.py
```

Restart `claude`, run `/mcp`, and both entries are listed: one from the project file, one
from your own. Then clean up, because this one follows you out of the project:

```bash
claude mcp remove --scope user devtools-personal
```

Stop here until after Chapter 7.